# Preparación de Datos para K-Means Clustering

Este notebook tiene como objetivo preparar un conjunto de datos de clientes para su uso en un modelo de clustering K-Means. A continuación se describe cada una de las celdas de código:

1. **Importación de Librerías y Carga de Datos**:
   - Se importan las librerías necesarias (`pandas`, `numpy`, `os`).
   - Se cargan los datos de clientes desde una URL pública de Google Drive.

2. **Conversión de Columnas de Fechas**:
   - Se convierten las columnas `registration_date` y `last_purchase_date` al tipo `datetime`.
   - Se valida la conversión imprimiendo los tipos de datos de las columnas.

3. **Preparación de Datos para K-Means**:
   - Se realizan varias transformaciones en los datos:
     - Conversión del valor medio de orden de USD a COP.
     - Cálculo del tiempo de vida del cliente en días.
     - Cálculo del valor histórico del cliente.
     - Cálculo de los días desde la última compra.
     - Cálculo de métricas adicionales como total de compras, frecuencia de compra, valor medio de orden, probabilidad de churn y valor del cliente.
   - Se seleccionan las columnas relevantes para el modelo de clustering.
   - Se imprimen las primeras filas de los datos preparados.

4. **Verificación de Valores Infinitos e Inválidos**:
   - Se verifica si existen valores infinitos en los datos preparados.
   - En caso de haber valores infinitos, se reemplazan por 0.5.
   - Se verifica si el conjunto de datos contiene algún valor inválido (nulos o infinitos).

5. **Exportación de Datos Preparados**:
   - Se crea el directorio de salida si no existe.
   - Se exportan los datos preparados a un archivo CSV.
   - Se imprime un mensaje confirmando la exportación exitosa.

## Importación de Librerías y Carga de Datos
En esta celda se importan las librerías necesarias y se cargan los datos de clientes desde una URL pública de Google Drive.

In [1]:
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np
import os

# DataSets Cargados publicamente desde drive
customers_url = "https://drive.google.com/uc?id=1KbGDXj8ruGIfJZ7iviui0DhKbrhJnttj"

# Leer los archivos desde las URLs
customers_df = pd.read_csv(customers_url)

## Conversión de Columnas de Fechas
En esta celda se convierten las columnas `registration_date` y `last_purchase_date` al tipo `datetime` y se valida la conversión imprimiendo los tipos de datos de las columnas.

In [2]:
# Fechas con formato objet
# No hay valores nulos
from datetime import datetime

# Convertir columnas de fechas a datetime
customers_df['registration_date'] = pd.to_datetime(customers_df['registration_date'], errors='coerce')
customers_df['last_purchase_date'] = pd.to_datetime(customers_df['last_purchase_date'], errors='coerce')

# Validar los cambios
print(customers_df.dtypes)

id                              int64
name                           object
registration_date      datetime64[ns]
last_purchase_date     datetime64[ns]
average_order_value           float64
total_purchases                 int64
churn_status                    int64
dtype: object


## Preparación de Datos para K-Means
En esta celda se realizan varias transformaciones en los datos de clientes para preparar el conjunto de datos que se utilizará en el modelo de clustering K-Means. Se convierten valores, se calculan métricas adicionales y se seleccionan las columnas relevantes.

In [3]:
# Convertir la columna 'average_order_value' de USD a COP
customers_df['average_order_value']=customers_df['average_order_value'] *4200 # promedio USD
# Calcular tiempo de vida del cliente (en mese)
customers_df['days_as_customer'] = (datetime.now() - customers_df['registration_date']).dt.days
customers_df['history_value'] = (customers_df['average_order_value']*customers_df['total_purchases']).round(2)
customers_df['dias_ultima_compra'] = (datetime.now() - customers_df['last_purchase_date']).dt.days

# 1. Total de Compras (total_compras)
customers_df['total_compras'] = customers_df['total_purchases']

# 2. Frecuencia de Compra (frecuencia_compra)
customers_df['frecuencia_compra'] = (customers_df['total_purchases'] / (customers_df['days_as_customer'] / 30)).round(2)

# 3. Valor Medio de Orden (valor_medio_orden)
customers_df['valor_medio_orden'] = customers_df['average_order_value']

# 4. Probabilidad de Churn (probabilidad_churn)
customers_df['probabilidad_churn'] = (customers_df['dias_ultima_compra']/((customers_df['frecuencia_compra']+1)*100)).round(2)

# 5. Calcular Clv (valor_cliente)
customers_df['valor_cliente']=(customers_df['history_value']/customers_df['days_as_customer']).round(2)

# Seleccionar columnas relevantes para el modelo
prepared_data = customers_df[[
    'id',
    'total_compras',
    'frecuencia_compra',
    'dias_ultima_compra',
    'valor_medio_orden',
    'probabilidad_churn',
    'valor_cliente'
]]
# Verificar las primeras filas
print("Datos preparados para K-Means Clustering:")
print(prepared_data.head())

Datos preparados para K-Means Clustering:
   id  total_compras  frecuencia_compra  dias_ultima_compra  \
0   1             17               0.47                 852   
1   2              6               0.07                1782   
2   3              5               0.16                 170   
3   4              5               0.12                 417   
4   5              1               0.01                1852   

   valor_medio_orden  probabilidad_churn  valor_cliente  
0           134442.0                5.80        2116.22  
1           365316.0               16.65         813.02  
2           188265.0                1.47        1014.36  
3           148092.0                3.72         578.03  
4           110279.4               18.34          39.33  


## Verificación de Valores Infinitos e Inválidos
En esta celda se verifica si existen valores infinitos en los datos preparados y, en caso de haberlos, se reemplazan por 0.5. Además, se verifica si el conjunto de datos contiene algún valor inválido (nulos o infinitos).

In [4]:
# Verificar valores infinitos
print("\n¿Existen valores infinitos?")
print(np.isinf(prepared_data).sum())

# En caso de haber ajustarlos:(solo si es necesario)
#prepared_data.replace([np.inf, -np.inf], .5, inplace=True)

# Verificar si el conjunto de datos tiene problemas generales
print("\n¿El conjunto de datos contiene algún valor inválido?")
print(prepared_data.isnull().values.any() or np.isinf(prepared_data).values.any())



¿Existen valores infinitos?
id                    0
total_compras         0
frecuencia_compra     0
dias_ultima_compra    0
valor_medio_orden     0
probabilidad_churn    0
valor_cliente         0
dtype: int64

¿El conjunto de datos contiene algún valor inválido?
False


## Exportación de Datos Preparados
En esta celda se crea el directorio de salida si no existe y se exportan los datos preparados a un archivo CSV. Se imprime un mensaje confirmando la exportación exitosa.

In [5]:
output_dir = os.path.join("Data")
os.makedirs(output_dir, exist_ok=True)

# Exportar los datos preparados
output_path = os.path.join(output_dir, "kmeans_m1_data.csv")
prepared_data.to_csv(output_path, index=False)
print(f"\nArchivo '{output_path}' guardado exitosamente.")


Archivo 'Data\kmeans_m1_data.csv' guardado exitosamente.
